#  MLP

*NOTE*: In this tutorial we follow Bengio et al. NN architecture

Sources:
1. video: https://www.youtube.com/watch?v=TCH_1BHY58I&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=3
2. paper: https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

In [1114]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as pt
%matplotlib inline

In [1115]:
words = open("../data/names.txt", "r").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [1116]:
# build characters list from the whole dataset (and more)
chars = sorted(list(set("".join(words))))  # all unique characters in the dataset
se_ch = "."  # start / end character

# build character to integer lookup table
# note: used to place the character at correct cell in the 2D array

# starting from integer 1 (0 is reserved)
stoi = {c: i + 1 for i, c in enumerate(chars)}
# start / end characters number is zero
stoi[se_ch] = 0

# build integer to character lookup table
# note: used to find the index of a character in the 2D array
itos = {i: c for c, i in stoi.items()}

In [1117]:
print(itos)
vocab_size = len(stoi)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [1118]:
# compile the dataset for the NN

block_size = 3  # context length: how many characters we take to predict next one

X = []  # inputs - indexes for 3 chars
Y = []  # labels - index for output 1 char

for w in words:
    context = [0] * block_size  # starting with ... (0 in vocab is '.', a dot)
    # print(w)

    w = w + "."  # add end char
    for ch in w:
        ix = stoi[ch]  # index lookup
        X.append(context)
        Y.append(ix)
        # next context; moving window through the word, one char at a time
        # print(f'{"".join([itos[c] for c in context])} ---> {ch}')
        context = context[1:] + [ix]

# Create tensors
X = torch.tensor(X)
Y = torch.tensor(Y)

In [1119]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [1120]:
# Creating embeddings lookup table for the characters vocabulary
seed = 2147483647
g = torch.Generator().manual_seed(seed)  # for deterministic output

# C lookup table is a vocab_size * dim size (e.g. 27 *2)
# Each line is an d items array of numbers - called embedding
# In other words, for each character in the vocabulary,
# we create a d size array of numbers

dim = 2  # embedding size
C = torch.randn((vocab_size, dim), generator=g)

In [1121]:
# Lookup embeddings
# We'll take 3 chars in our NN as an input
# So for these 3 chars we need to input 3 embeddings simultaneously
# This why the emb exists - shape [vocab_size, input_size, embedding_size], e.g. [300K, 3, 2]
emb = C[X]
emb.shape, C.shape, X.shape

(torch.Size([228146, 3, 2]), torch.Size([27, 2]), torch.Size([228146, 3]))

In [1122]:
# NN

# Note: We need to input all 3 characters once in our NN
# For that we can concatenate the 3 character embeddings into a single vector
# So we can interpret our input vector, as first layer of NN, thus W notation
# input size, e.g. 6 because: 3 input chars * 2 embedding size

input_size = block_size * dim  # inputs size
h_size = 100  # hidden layer outputs size
W1 = torch.randn((input_size, h_size))  # input_size input, 100 output
b1 = torch.randn(h_size)  # bias

In [1123]:
# forward pass

# Note: Because of different dimensionality,
# we cannot multiply and add '@' in the first layer, e.g. emb @ W1 + b
# For that we need to 'arrange' the shapes

# We have:
# emb.shape, W1.shape
# Note: 32 here is 32 inputs, just a fraction of the input sequence,
# for easier development. Normally it's the size off all 3 char inputs from dataset.
# (torch.Size([32, 3, 2]), torch.Size([6, 100]))

# We want:
# [32, 6], [6, 100]

# One way torch.cat - concatenate.
# But this is not scalable.
# If we're to increase the number of input characters, this code breaks
ex_cat = torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)  # [32, 6]

# Another way with unbind, more scalable
ex_cat_unbind = torch.cat(torch.unbind(emb, 1), 1)  # [32, 6] same as above

# Better, more efficient way is by doing it with view.
# .view doesn't create a new tensor in the memory (the other 2 do)
ex_view = emb.view(-1, input_size)  # !Our winner

ex_cat.shape, ex_cat_unbind.shape, ex_view.shape

(torch.Size([228146, 6]), torch.Size([228146, 6]), torch.Size([228146, 6]))

In [1124]:
# check all 3 are equal
(ex_cat == ex_cat_unbind) & (ex_cat_unbind == ex_view)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        ...,
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True]])

In [1125]:
# first layer
# h = torch.tanh(emb.view(emb.shape[0], input_size) @ W1 + b1)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h.shape, h

(torch.Size([228146, 100]),
 tensor([[-0.6221, -0.9999,  0.7302,  ..., -0.9749, -0.9958, -0.9952],
         [ 0.7299, -0.9181,  0.9587,  ..., -0.9768, -0.9746, -1.0000],
         [ 0.3370, -1.0000,  0.8726,  ...,  0.8768, -0.3990,  0.9800],
         ...,
         [ 0.9074,  0.9142,  0.9419,  ..., -0.5155, -0.0420, -0.8355],
         [ 0.8907, -0.1010,  0.4699,  ...,  0.9975,  0.7528,  0.9945],
         [ 0.6271, -0.9008,  0.2521,  ..., -0.5059, -0.6785, -0.9965]]))

In [1126]:
# second layer
W2 = torch.randn((100, 27))
b2 = torch.rand(27)

In [1127]:
logits = h @ W2 + b2
logits.shape

torch.Size([228146, 27])

In [1128]:
counts = logits.exp()

In [1129]:
prob = counts / counts.sum(1, keepdim=True)
prob.shape

torch.Size([228146, 27])

In [1130]:
loss = -prob[torch.arange(len(emb)), Y].log().mean()
loss

tensor(15.4313)

In [1131]:
# --- make it respectable ---#

In [1132]:
X.shape, Y.shape  # dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [1133]:
g = torch.Generator().manual_seed(2147483647)  # for determinism

# NN
C = torch.randn((vocab_size, dim), generator=g, requires_grad=True)
W1 = torch.randn((input_size, h_size), generator=g, requires_grad=True)
b1 = torch.randn(h_size, generator=g, requires_grad=True)
W2 = torch.randn((h_size, vocab_size), generator=g, requires_grad=True)
b2 = torch.randn(vocab_size, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [1134]:
# number of parameters in total
print("parameters:", sum(p.nelement() for p in parameters))

parameters: 3481


In [1135]:
for p in parameters:
    p.requires_grad = True

In [1146]:
for _ in range(1000):
    
    # mini-batch construct (for efficiency)
    ix = torch.randint(0, X.shape[0], (32, )) # 32 random indexes from X; shape (32)
    
    # forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, input_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

print('loss:', loss.item())

loss: 2.6770668029785156


In [1147]:
emb = C[X]
h = torch.tanh(emb.view(-1, input_size) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(2.6532, grad_fn=<NllLossBackward0>)